<a href="https://colab.research.google.com/github/HarshLogic/Employee-Attrition-Prediction/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Data Loading & Exploration**

##  **Load the dataset**

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [19]:
file_path = 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(file_path)

In [20]:
print("First 10 rows")
df.head(10)

First 10 rows


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
5,32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,8,...,3,80,0,8,2,2,7,7,3,6
6,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,1,80,3,12,3,2,1,0,0,0
7,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,2,80,1,1,2,3,1,0,0,0
8,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,...,2,80,0,10,2,3,9,7,1,8
9,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,...,2,80,2,17,3,2,7,7,7,7


In [21]:
rows, col = df.shape
print(f"\nDimination: {rows} rows, {col} col")


Dimination: 1470 rows, 35 col


In [22]:
target_col = 'Attrition'
counts = df[target_col].value_counts()
print(f"\nAttrition counts:\n{counts}")


Attrition counts:
Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [23]:
attrition_rate = (counts['Yes'] / rows) * 100
print(f"\nAttrition rate: {attrition_rate:.2f}%")


Attrition rate: 16.12%


In [24]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print(f"\nNumeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")


Numeric columns: 26
Categorical columns: 9


Observation: The dataset is imbalanced. Only ~16% of employees have left, meaning the model will have significantly more examples of employees who stayed, which could lead to bias toward predicting "No".

## Data Cleaning & Preprocessing

### Handle Nulls

In [25]:
df = df.dropna()

### Drop irrelevant columns

In [26]:
cols_to_drop = ['EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df = df.drop(columns=cols_to_drop)

### Encode Target

In [7]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

### Encode Categorical & Scale

In [8]:
from sklearn.preprocessing import StandardScaler
df = pd.get_dummies(df, drop_first=True)

### Separate features/target and Scale

In [11]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

##**Exploratory Data Analysis (EDA)**

In [12]:
# Convert Attrition to numeric for calculation: Yes=1, No=0
df['Attrition_Num'] = df['Attrition'].map({'Yes': 1, 'No': 0})

### 1. **Attrition by Department**

In [33]:
df['Attrition_Num'] = df['Attrition'].map({'Yes': 1, 'No': 0})
dept_attrition = df.groupby('Department')['Attrition_Num'].mean() * 100
print("Attrition Rate by Department:\n", dept_attrition)

Attrition Rate by Department:
 Department
Human Resources           19.047619
Research & Development    13.839750
Sales                     20.627803
Name: Attrition_Num, dtype: float64


#**Model Building & Evaluation**

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier()
}

In [32]:
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "AUC": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
        "F1-Score": classification_report(y_test, preds, output_dict=True)['1']['f1-score']
    })
print(pd.DataFrame(results))

                 Model       AUC  F1-Score
0  Logistic Regression  0.765913  0.356589
1        Random Forest  0.728859  0.142857
2    Gradient Boosting  0.779789  0.269231
